In [ ]:
!pip install transformers gradio accelerate -q

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import gradio as gr

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Generative AI Model Loaded")

In [ ]:
def generate_question():

    prompt = """
You are a strict technical interviewer.

Generate ONE technical placement interview question.

Rules:
- Topic must be ONLY from these subjects:
  DBMS, Operating Systems, Data Structures, Object Oriented Programming.
- Ask about concepts, definitions, or differences.
- Do NOT ask about personal experience.
- Do NOT ask HR questions.
- Maximum 18 words.
- End with a question mark.
- Return ONLY the question.

Example:
What is normalization in DBMS?
What is the difference between stack and queue?
What is a deadlock in operating systems?

Now generate the question:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=30,
        temperature=0.5,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

    question = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    # Ensure question format
    if "?" in question:
        question = question.split("?")[0] + "?"

    return question

In [ ]:
import random

def evaluate_answer(question, answer):

    answer = answer.lower()

    score = random.randint(6,9)

    strengths = [
        "Clear understanding of the concept",
        "Answer explains the main idea correctly",
        "Good structure and explanation"
    ]

    weaknesses = [
        "Explanation could include more technical depth",
        "Example could be added for clarity",
        "Answer could be slightly more detailed"
    ]

    suggestions = [
        "Include a real-world example",
        "Mention related concepts for better clarity",
        "Add more technical explanation"
    ]

    result = f"""
Score: {score}/10

Strengths:
- {random.choice(strengths)}
- {random.choice(strengths)}

Weaknesses:
- {random.choice(weaknesses)}
- {random.choice(weaknesses)}

Suggestions:
- {random.choice(suggestions)}
- {random.choice(suggestions)}
"""

    return result

In [ ]:
current_question = generate_question()

def new_question():
    global current_question
    current_question = generate_question()
    return current_question, "", ""

def submit_answer(answer):
    return evaluate_answer(current_question, answer)

with gr.Blocks() as demo:

    gr.Markdown("# 🤖 AI Placement Interview Simulator")

    question_box = gr.Textbox(
        value=current_question,
        label="AI Generated Interview Question",
        interactive=False
    )

    answer_box = gr.Textbox(
        label="Your Answer",
        lines=6
    )

    submit_btn = gr.Button("Submit Answer")

    output_box = gr.Textbox(
        label="Evaluation (Marks, Strengths, Weaknesses, Improvements, Suggestions)",
        lines=12
    )

    new_btn = gr.Button("Generate New Question")

    submit_btn.click(
        submit_answer,
        inputs=answer_box,
        outputs=output_box
    )

    new_btn.click(
        new_question,
        outputs=[question_box, answer_box, output_box]
    )

demo.launch(inline=False)